# Notebook 02 – Velozähldaten laden und verarbeiten
**Charge & Ride Züri** | ZHAW Geomarketing FS2026

## Zwei Datenquellen, zwei Abdeckungsgebiete

| | Stadt Zürich | Kanton Zürich |
|---|---|---|
| Zählstellen | ~25 | 97 |
| Abdeckung | Nur Stadtgebiet | Ganzer Kanton |
| Verfügbar seit | 2009 | 2016 |
| Granularität | Viertelstunden | Stunden/Tage |
| Formate | CSV, **Parquet**, PDF | CSV (manuell) |
| Quelle | data.stadt-zuerich.ch | zh.ch Dataset 3062 |

**Strategie:** Kantonsdaten für die räumliche Hauptanalyse (ganzer Kanton),
Stadtdaten für höhere Auflösung im Stadtgebiet.

---

### Daten-URLs (alle verifiziert, CC-Zero / OGD)

**Stadt ZH – Standorte (GeoJSON mit Koordinaten):**
```
https://www.stadt-zuerich.ch/geodaten/download/Standorte_der_automatischen_Fuss__und_Velozaehlungen
```
Formate: CSV, dxf, gpkg, **json (GeoJSON)**, shp, wfs, wms, wmts

**Stadt ZH – Frequenzdaten (Viertelstundenwerte):**
```
https://data.stadt-zuerich.ch/dataset/ted_taz_verkehrszaehlungen_werte_fussgaenger_velo
```
Formate: CSV, **Parquet**, PDF (Korrekturfaktoren)

**Parquet-Download 2025 (deutlich schneller als CSV):**
```
https://data.stadt-zuerich.ch/dataset/ted_taz_verkehrszaehlungen_werte_fussgaenger_velo/download/2025_verkehrszaehlungen_werte_fussgaenger_velo.parquet
```

**Kanton ZH – Velozähldaten (OGD Dataset 3062):**
```
https://www.zh.ch/de/politik-staat/statistik-daten/datenkatalog.html#/datasets/3062@tiefbauamt-kanton-zuerich
```
→ Manueller Download, dann in data/raw/ ablegen

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily as cx

ROOT = Path('..').resolve()
DATA_RAW = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Direkte Download-URLs Stadt ZH
BASE_URL = 'https://data.stadt-zuerich.ch/dataset/ted_taz_verkehrszaehlungen_werte_fussgaenger_velo/download/'
URL_PARQUET_2025 = BASE_URL + '2025_verkehrszaehlungen_werte_fussgaenger_velo.parquet'
URL_CSV_2024     = BASE_URL + '2024_verkehrszaehlungen_werte_fussgaenger_velo.csv'

# GeoJSON-Standorte Stadt ZH
URL_STANDORTE_GEOJSON = (
    'https://www.stadt-zuerich.ch/geodaten/download/'
    'Standorte_der_automatischen_Fuss__und_Velozaehlungen?format=geojson'
)

print('Setup abgeschlossen.')

## TEIL A: Frequenzdaten Stadt Zürich laden

Wir laden das **Parquet-Format** – es ist 5-10x schneller als CSV
und benötigt weniger Speicher. Fallback auf CSV falls Parquet fehlt.

In [ ]:
# Parquet laden (bevorzugt – schneller und kompakter als CSV)
print('Lade Frequenzdaten (Parquet)...')
try:
    df_freq = pd.read_parquet(URL_PARQUET_2025)
    print(f'Parquet geladen: {df_freq.shape[0]:,} Zeilen, {df_freq.shape[1]} Spalten')
    print(f'Speicherbedarf: {df_freq.memory_usage(deep=True).sum() / 1e6:.1f} MB')
    DATEN_QUELLE = 'Parquet 2025'
except Exception as e:
    print(f'Parquet nicht verfuegbar ({e}), lade CSV als Fallback...')
    df_freq = pd.read_csv(URL_CSV_2024)
    DATEN_QUELLE = 'CSV 2024'
    print(f'CSV geladen: {df_freq.shape[0]:,} Zeilen')

print(f'\nSpalten: {df_freq.columns.tolist()}')
print(f'Datentypen:')
print(df_freq.dtypes)
df_freq.head(3)

In [ ]:
# Datenstruktur erkunden
# WICHTIG: Das CSV/Parquet enthaelt SOWOHL Velo- ALS AUCH Fussgaengerzaehlstellen!

print('=== Spaltenstruktur ===')
for col in df_freq.columns:
    n_unique = df_freq[col].nunique()
    if n_unique < 30:
        print(f'  {col} ({n_unique} Werte): {sorted(df_freq[col].dropna().unique().tolist())[:10]}')
    else:
        print(f'  {col}: {n_unique} eindeutige Werte (Beispiel: {df_freq[col].iloc[0]})')

# Hinweis: Der Join-Schluessel heisst FK_ZAEHLER im Datensatz
# und verbindet mit dem Geodatensatz

In [ ]:
# Nur Velozaehlstellen behalten (Fussgaenger herausfiltern)
# Typischer Filterwert: 'Velo' oder 'V' – nach obiger Ausgabe anpassen!

# Spaltenname fuer den Typ-Filter (nach oben stehender Ausgabe anpassen):
TYP_SPALTE = 'Typ'          # <- anpassen falls anders
VELO_TYP_WERT = 'Velo'      # <- anpassen falls anders (z.B. 'FV', 'V')

if TYP_SPALTE in df_freq.columns:
    df_velo = df_freq[df_freq[TYP_SPALTE] == VELO_TYP_WERT].copy()
    print(f'Gesamt: {len(df_freq):,} Zeilen -> Velo: {len(df_velo):,} Zeilen')
else:
    df_velo = df_freq.copy()
    print('WARNUNG: Kein Typ-Feld gefunden. Alle Zeilen werden verwendet.')

# Datum parsen
df_velo['Datum'] = pd.to_datetime(df_velo['Datum'])
df_velo['datum_tag'] = df_velo['Datum'].dt.date

print(f'Zeitraum: {df_velo["Datum"].min()} bis {df_velo["Datum"].max()}')
print(f'Zaehlstellen: {df_velo["FK_ZAEHLER"].nunique()}')

In [ ]:
# Mittlere Tagesfrequenz pro Zaehlstelle berechnen
# (Summe aller Viertelstunden eines Tages, dann Mittelwert ueber alle Tage)

# Wertspalte ermitteln (heisst 'Total', 'Anzahl', 'Count' o.ae.)
WERT_SPALTE = 'Total'  # <- anpassen nach oben stehender Spaltenausgabe

# Tagesfrequenz: alle 96 Viertelstunden eines Tages summieren
df_daily = (
    df_velo
    .groupby(['FK_ZAEHLER', 'datum_tag'])[WERT_SPALTE]
    .sum()
    .reset_index()
)

# Mittlere Tagesfrequenz (DTV = Durchschnittlicher Tagesverkehr) pro Zaehlstelle
df_dtv = (
    df_daily
    .groupby('FK_ZAEHLER')[WERT_SPALTE]
    .agg(dtv_velos='mean', max_tag='max', anzahl_tage='count')
    .reset_index()
)

print('Top-10 Zaehlstellen nach mittlerem Tagesverkehr (DTV):')
print(df_dtv.sort_values('dtv_velos', ascending=False).head(10).to_string(index=False))

## TEIL B: Standorte (GeoJSON) laden und mit Frequenzen joinen

In [ ]:
# Standorte als GeoJSON laden
# Formate verfuegbar: csv, dxf, gpkg, json (GeoJSON!), shp, wfs, wms, wmts
# Quelle: https://www.stadt-zuerich.ch/geodaten/download/Standorte_der_automatischen_Fuss__und_Velozaehlungen

try:
    gdf_standorte = gpd.read_file(URL_STANDORTE_GEOJSON)
    print(f'Standorte geladen: {len(gdf_standorte)} Eintraege')
    print(f'CRS: {gdf_standorte.crs}')
    print(f'Spalten: {gdf_standorte.columns.tolist()}')
except Exception as e:
    print(f'Online-Laden fehlgeschlagen: {e}')
    print('Bitte Standorte-Datei manuell als GeoJSON herunterladen:')
    print('URL: https://www.stadt-zuerich.ch/geodaten/download/Standorte_der_automatischen_Fuss__und_Velozaehlungen')
    # gdf_standorte = gpd.read_file(DATA_RAW / 'standorte_velozaehlungen_stzh.json')

In [ ]:
# Join: Standorte + DTV-Frequenzen
# Join-Schluessel: FK_ZAEHLER im GeoJSON = FK_ZAEHLER im CSV/Parquet

gdf_standorte['FK_ZAEHLER'] = gdf_standorte['FK_ZAEHLER'].astype(str)
df_dtv['FK_ZAEHLER'] = df_dtv['FK_ZAEHLER'].astype(str)

gdf_mit_dtv = gdf_standorte.merge(df_dtv, on='FK_ZAEHLER', how='left')

# In LV95 umrechnen fuer Distanzberechnungen in Metern
gdf_mit_dtv = gdf_mit_dtv.to_crs(epsg=2056)

print(f'Zaehlstellen: {len(gdf_mit_dtv)}')
print(f'Davon mit DTV-Daten: {gdf_mit_dtv["dtv_velos"].notna().sum()}')

# Als GeoJSON speichern (in WGS84 fuer GeoJSON-Standard)
gdf_mit_dtv.to_crs(epsg=4326).to_file(
    DATA_PROCESSED / 'velozaehlstellen_stadt_zh.geojson',
    driver='GeoJSON'
)
print('Gespeichert: data/processed/velozaehlstellen_stadt_zh.geojson')

## TEIL C: Kantonale Velozählstellen (Dataset 3062)

**Einmaliger manueller Download-Schritt:**
1. Gehe zu: `https://www.zh.ch/de/politik-staat/statistik-daten/datenkatalog.html#/datasets/3062@tiefbauamt-kanton-zuerich`
2. Datensatz herunterladen → speichere in `data/raw/`
3. Dateiname unten entsprechend setzen

In [ ]:
# Kantonale Rohdaten einlesen (nach manuellem Download)
kanton_files = list(DATA_RAW.glob('velozaehlstellen_kanton*'))
print(f'Gefundene Kantons-Dateien: {kanton_files}')

if not kanton_files:
    print('HINWEIS: Kantonale Daten noch nicht heruntergeladen.')
    print('Bitte von zh.ch Dataset 3062 herunterladen.')
    print('GIS-Browser: https://geo.zh.ch/maps?initialMapIds=TBAVMSZH')
else:
    f = kanton_files[0]
    if f.suffix in ['.geojson', '.json']:
        gdf_kanton = gpd.read_file(f).to_crs(epsg=2056)
    elif f.suffix == '.csv':
        df_kanton = pd.read_csv(f)
        print(f'Spalten: {df_kanton.columns.tolist()}')
        # GeoDataFrame aus X/Y-Koordinaten erstellen (Spaltennamen anpassen!)
        # gdf_kanton = gpd.GeoDataFrame(
        #     df_kanton,
        #     geometry=gpd.points_from_xy(df_kanton['X'], df_kanton['Y']),
        #     crs='EPSG:2056'
        # )
    print(f'Kantonsdata geladen: OK')

## Visualisierung: Alle Zählstellen auf einen Blick

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

# Kantonsgrenze
gdf_kanton_boundary = gpd.read_file(DATA_PROCESSED / 'kanton_zh_boundary.geojson').to_crs(epsg=2056)
gdf_kanton_boundary.boundary.plot(ax=ax, color='black', linewidth=1.5, zorder=3)

# Stadt ZH Zaehlstellen (farbig nach DTV)
gdf_mit_dtv.plot(
    ax=ax,
    column='dtv_velos',
    cmap='YlOrRd',
    markersize=80,
    legend=True,
    legend_kwds={'label': 'DTV Velos/Tag (Stadt ZH)'},
    zorder=5,
    missing_kwds={'color': 'grey', 'alpha': 0.5}
)

cx.add_basemap(ax, crs='EPSG:2056', source=cx.providers.OpenStreetMap.Mapnik, alpha=0.4)
ax.set_title('Velozaehlstellen Kanton Zuerich\n(Rot = hohe Frequenz, Stadtgebiet)', fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'karte1_velozaehlstellen_frequenz.png', dpi=150, bbox_inches='tight')
plt.show()
print('Karte 1 gespeichert (Basis fuer QGIS-Layout).')

## Zusammenfassung

| Datensatz | Format geladen | Zählstellen | Output |
|---|---|---|---|
| Stadt ZH Frequenzen | Parquet (schnell!) | 25 | velozaehlstellen_stadt_zh.geojson |
| Kanton ZH | CSV/GeoJSON (manuell) | 97 | (nach Download) |

→ Weiter mit **Notebook 03** (Scoring-Modell)